# SCM-VLM: train on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Checkpoints, the feature cache and results are stored in your Google Drive (`MyDrive/scm_vlm/`),
so nothing is lost if Colab disconnects. The dataset (~13 GB) is downloaded fresh each session
(about 5-10 minutes).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive (keeps checkpoints between sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/scm_vlm'
import os
for sub in ['checkpoints', 'cache', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

## 3. Get the code

Push your latest code to GitHub first (from your laptop: `git add -A`, `git commit`, `git push`).
If the repository is **private**, replace the URL with
`https://<your-github-token>@github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git`
(create a token at GitHub → Settings → Developer settings → Personal access tokens).

In [ ]:
REPO = 'https://github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git'
%cd /content
!rm -rf scm && git clone -q $REPO scm
%cd /content/scm
!git log --oneline | head -3

## 4. Install dependencies (PyTorch is already installed on Colab)

In [ ]:
!pip install -q transformers timm networkx seaborn openai

## 5. Keep checkpoints / cache / results on Drive

In [ ]:
%cd /content/scm
!rm -rf checkpoints cache results
!ln -s $DRIVE_DIR/checkpoints checkpoints
!ln -s $DRIVE_DIR/cache cache
!ln -s $DRIVE_DIR/results results
!ls -l | grep -- '->'

## 6. Download VQA v2 + COCO train2014 (~13 GB, 5-10 min)

The COCO object annotations (`instances_train2014.json`) are also downloaded; they are needed
for the counterfactual-supervision step later.

If a download fails (404/403), get the current links from https://visualqa.org/download.html (*Training questions* and *Training annotations*, VQA v2) and https://cocodataset.org/#download (*2014 Train images* and *2014 Train/Val annotations*), and paste them into the cell below.

In [ ]:
%cd /content/scm
!mkdir -p coco/vqa coco/train2014 coco/annotations
# VQA v2 questions + answers
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip -O /content/q.zip
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip -O /content/a.zip
!unzip -q -o /content/q.zip -d coco/vqa && unzip -q -o /content/a.zip -d coco/vqa
# COCO object annotations
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2014.zip -O /content/ann.zip
!unzip -q -o /content/ann.zip -d coco
# COCO train2014 images  ->  coco/train2014/train2014/*.jpg
!wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip -O /content/train2014.zip
!unzip -q -o /content/train2014.zip -d coco/train2014 && rm /content/train2014.zip
!ls coco/vqa coco/annotations | head; ls coco/train2014/train2014 | wc -l

## 7. Check that questions and images match

In [ ]:
!python validate_dataset.py

## 8. Train

Short run first (step 1 of the plan). The first epoch runs DETR on every image and fills the cache
on Drive; later epochs (and later sessions) reuse it and are much faster.

For the full run later: `!python train.py --samples 20000 --epochs 10`

In [ ]:
!python train.py --samples 3000 --epochs 5

In [ ]:
import pandas as pd
display(pd.read_csv('results/training_history.csv'))
display(pd.read_csv('results/final_metrics.csv'))

## 9. Try the trained model on any image

In [ ]:
!python main.py --image images/Man_riding_bicycle.png --question "What is the man riding?"
!python whatif_qa.py --image images/Man_riding_bicycle.png --whatif "what if the bicycle were a motorcycle, what is the man riding?"

To use your own picture: upload it with the folder icon on the left (into `/content/scm/images/`),
then change the `--image` path above.

## 10. Copy the checkpoint to your laptop

It is already in Google Drive at `MyDrive/scm_vlm/checkpoints/scm_vlm_latest.pt`.
Download it from Drive and put it in the project's `checkpoints/` folder, or run the cell below.

In [ ]:
from google.colab import files
files.download('checkpoints/scm_vlm_latest.pt')